# Suite BRV — The Verilog bridge

`Bridges.Verilog` carries expressions between mbse-expressions' SystemVerilog dialect and the Verilog language's syntax
trees, both ways, tree to tree. Each case writes terms as syntax nodes, whose printed SystemVerilog is the dialect's
own render, and reads syntax nodes, parsed from SystemVerilog source, as terms. Constraints written as functions are
checked against the dialect's evaluator, and constraints from mbse-expressions' Basic dialect round-trip through it.

In [ ]:
import { Evaluators, Expressions as E, Text } from "@mbse/expressions/Dialects/SystemVerilog";
import { Terms as F } from "@mbse/expressions/Framework";
import { Verilog as B } from "@mbse/programs/Bridges";
import { Errors } from "@mbse/programs/Framework";
import { SystemVerilog2023, Syntax as S } from "@mbse/programs/Verilog";
import { Evaluators as BasicEvaluators, Expressions as Basic } from "@mbse/expressions";
import * as Translators from "@mbse/expressions/Translators";
import { Proxies, Schemas as SC } from "@mbse/schemas/Framework";

import { assert, equal, raises } from "./support.js";

const printed = (term: any) => SystemVerilog2023.print(B.expression(term));

/** The functions that return the expressions `texts`, parsed from one source. */
function functions(...texts: string[]): S.FunctionDeclaration[] {
  const source = texts.map((text, i) => `function automatic logic f${i}(); return ${text}; endfunction\n`).join("");
  return SystemVerilog2023.parse(source).items as S.FunctionDeclaration[];
}

const read = (...texts: string[]) => functions(...texts).map((fn) => B.term_of_function(fn));

/** Reading each [text, message]'s text fails with its message. */
function fails(...cases: [string, string][]): void {
  functions(...cases.map(([text]) => text)).forEach((fn, i) => {
    raises(Errors.TranspileError, () => B.term_of_function(fn), cases[i]![1]);
  });
}

## BRV-01 · Every term of the dialect writes the SystemVerilog the dialect renders, and reads back the same

In [ ]:
const [x, y, z, q] = ["x", "y", "z", "q"].map((name) => E.identifier(name));
const TERMS: any[] = [
  E.constant(5n), E.vector("10x1", null, "b"), E.vector("11111111", true, "h"), E.vector("0101", null, "o"),
  E.vector("1100", null, "d"), E.vector("zzzz", null, "d"), E.unary("~", x), E.unary("!", E.binary("==", x, 1n)),
  E.binary("-", x, E.binary("-", y, z)), E.binary("**", E.unary("-", x), 2n), E.binary("->", x, E.binary("->", y, z)),
  E.binary("<->", E.binary("->", x, y), z), E.binary("==?", x, E.vector("1zz0", null, "b")), E.binary("<<<", x, 2n),
  E.conditional(x, 1n, 2n), E.conditional(E.conditional(x, 1n, 2n), 3n, 4n), E.conditional(x, y, E.conditional(y, 1n, 2n)),
  E.concatenation(x, y, E.vector("01", null, "b")), E.replication(4n, x), E.replication(2n, E.concatenation(x, y)),
  E.select(x, 3n), E.range_(x, 7n, 0n), E.select(E.member(x, "data"), E.binary("+", y, 1n)),
  E.inside(x, 1n, E.span(3n, 5n), y), E.binary("&&", E.inside(x, 1n), y), E.unary("!", E.inside(x, 1n)),
  E.cast("int", x), E.cast("signed", x), E.cast(8n, x), E.cast("real", x), E.cast("logic", x), E.cast("unsigned", y),
  E.member(E.member(x, "a"), "b"), E.member(E.identifier("this"), "a"), E.call("$clog2", 8n), E.call("f", x, 1n),
  E.call("pkg.f"), E.call("this.f", x), E.method(q, "size"), E.method(q, "and"), E.method(E.member(x, "q"), "max"),
  E.iterate(q, "sum", "p", E.binary(">", E.identifier("p"), 0n)), E.iterate(q, "xor", "item", E.identifier("item")),
  E.binary("+", E.method(q, "size"), 1n), E.constant("a\"b\\c\n\t"), E.constant(2.5), E.unary("-", E.unary("-", x)),
  E.unary("&", E.unary("&", x)),
];
for (const term of TERMS) equal(printed(term), Text.ToText(term));
read(...TERMS.map(printed)).forEach((back, i) => assert(F.same(back, TERMS[i]), printed(TERMS[i])));
equal(printed(E.unary("-", E.unary("-", x))), "-(-x)"); // a unary operator's operand is a primary (IEEE 1800, A.8.3)
// A replication of several items repeats their concatenation.
assert(F.same(read("{2{x, y}}")[0], E.replication(2n, E.concatenation(x, y))));
assert(B.expression(E.identifier("this")) instanceof S.ThisExpression); // SystemVerilog's keyword
raises(Errors.TranspileError, () => B.expression(E.span(1n, 2n)), "a span is not an expression"); // but an item of `inside`
console.log(TERMS.length, "terms");

## BRV-02 · Constants: an unsized decimal, a real or a string, a negative number a negation, and back

In [ ]:
const CONSTANTS: [unknown, string][] = [
  [0n, "0"], [-1n, "-1"], [1180591620717411303424n, "1180591620717411303424"], [1.5, "1.5"], [-2.5, "-2.5"],
  [-0, "-0.0"], [1e16, "1e+16"], [1e-05, "1e-05"], [Infinity, "1.0 / 0.0"], [-Infinity, "-1.0 / 0.0"],
  [NaN, "0.0 / 0.0"], ["a\"b\\c\n\t", "\"a\\\"b\\\\c\\n\\t\""], ["é", "\"é\""], ["", "\"\""],
];
read(...CONSTANTS.map(([, text]) => text)).forEach((back, i) => {
  const [value, text] = CONSTANTS[i]!;
  const term = E.constant(value as any);
  equal(printed(term), text);
  equal(Text.ToText(term), text);
  equal(Text.ToText(back), text); // a negation and a division read as written
});
assert(F.same(read("-1")[0], E.unary("-", 1n)) && F.same(read("1.0 / 0.0")[0], E.binary("/", 1.0, 0.0)));
equal(printed(E.binary("*", E.identifier("x"), -1n)), "x * -1");
console.log("ok");

## BRV-03 · Literals are decoded as IEEE 1800 reads them: sized literals' bits, extended or truncated, and escapes

In [ ]:
const LITERALS: [string, string][] = [
  ["8'b1?0z", "8'b00001z0z"], ["4'hx", "4'hx"], ["4'h1z", "4'hz"], ["3'hff", "3'h7"], ["12'o7x", "12'o007x"],
  ["8'sd255", "8'sd255"], ["8'dz", "8'dz"], ["4'D?", "4'dz"], ["2'd7", "2'd3"], ["16'h_ff_x", "16'h0ffx"],
  ["8'HFF", "8'hff"], ["4'sb1", "4'sb0001"], ["8'bx1", "8'bxxxxxxx1"], ["8'oz", "8'bzzzzzzzz"],
  ["12_345", "12345"], ["1.5e3", "1500.0"], ["1_0.5", "10.5"],
];
read(...LITERALS.map(([text]) => text)).forEach((back, i) => equal(Text.ToText(back), LITERALS[i]![1]));
assert(F.same(B.decode(new S.IntegerLiteral({ spelling: "8'hFF" })), E.vector("11111111", null, "h")));
equal(B.decode(new S.IntegerLiteral({ spelling: "8'sd255" })).signed, true);
const STRINGS: [string, string][] = [
  ["\"\\101\\x41\\x4g\"", "AA\x04g"], ["\"\\n\\t\\\\\\\"\\v\\f\\a\"", "\n\t\\\"\v\f\x07"], ["\"\\q\\0\"", "q\0"],
  ["\"a\\\nb\"", "ab"],
];
read(...STRINGS.map(([text]) => text)).forEach((back, i) => equal(back.value, STRINGS[i]![1]));
// What slang refuses: a backslash that ends the text, and `\x` without digits.
for (const [text, value] of [["a\\", "a\\"], ["\\xg", "xg"]]) equal(B.decode(new S.StringLiteral({ text })).value, value);
console.log("ok");

## BRV-04 · A constraint becomes a function, a `constraint` block or an assertion, and each reads back to it

In [ ]:
const constraint = E.binary("&&", E.binary(">=", E.identifier("age"), 18n), E.binary(">", E.method(E.identifier("email"), "size"), 0n));
const contactable = B.function_("is_contactable", [["int", "age"], ["byte", "email"]], constraint);
equal(SystemVerilog2023.print(contactable),
  "function automatic logic is_contactable(input int age, input byte email);\n"
  + "    return age >= 18 && email.size() > 0;\nendfunction");
equal(SystemVerilog2023.print(B.function_("half", [["real", "x"]], E.binary("/", E.identifier("x"), 2.0), "real")),
  "function automatic real half(input real x);\n    return x / 2.0;\nendfunction");
equal(SystemVerilog2023.print(B.constraint("c_age", constraint)), "constraint c_age {\n    age >= 18 && email.size() > 0;\n}");
equal(SystemVerilog2023.print(B.assertion(constraint)), "assert (age >= 18 && email.size() > 0);");
equal(SystemVerilog2023.print(B.assertion(constraint, "not contactable")),
  "assert (age >= 18 && email.size() > 0) else $error(\"not contactable\");");
assert(F.same(B.term_of_function(contactable), constraint) && F.same(B.term_of_constraint(B.constraint("c", constraint)), constraint));
assert(F.same(B.term_of_assertion(B.assertion(constraint, "no")), constraint));
const declared: any = SystemVerilog2023.parse(`class person;
    rand int age;
    // adults only
    constraint c_age { /* the constraint */ age >= 18 && age < 150; }
endclass
module m;
    initial assert (8'hff == 255) else $error("no");
    function automatic logic f(input int x);
        // the constraint
        return x inside {[1:3], 5};
    endfunction
endmodule
`);
equal(Text.ToText(B.term_of_constraint(declared.items[0].items[2])), "age >= 18 && age < 150"); // after the comment
equal(Text.ToText(B.term_of_assertion(declared.items[1].items[0].body)), "8'hff == 255");
equal(Text.ToText(B.term_of_function(declared.items[1].items[1])), "x inside {[1:3], 5}");
const results = ([[20n, [1n]], [17n, [1n]], [30n, []]] as [bigint, bigint[]][])
  .map(([age, email]) => Evaluators.OfAny(constraint, { age, email }) as any);
equal(results.map((result) => result.integer()), [1n, 0n, 0n]); // the constraint, evaluated by the dialect
console.log("ok");

## BRV-05 · What the dialect cannot hold fails, at the syntax node's path

In [ ]:
const ITERATION = "an iteration is a reduction (sum, product, and, or, xor) of one iterator";
const CAST = "a cast names a built-in type, a signedness or a width";
fails(
  ["'b101", "body[0].value: an unsized based literal has no counterpart in the dialect"],
  ["'1 + 1", "body[0].value.left: UnbasedUnsizedLiteral has no counterpart in the SystemVerilog dialect"],
  ["x[3+:2]", "body[0].value: a part-select with +: has no counterpart in the dialect"],
  ["x[3-:2]", "body[0].value: a part-select with -: has no counterpart in the dialect"],
  ["x inside {[1 +/- 2]}", "body[0].value.set[0]: a tolerance range has no counterpart in the dialect"],
  ["my_t'(x)", `body[0].value.type: ${CAST}`], ["logic [3:0]'(x)", `body[0].value.type: ${CAST}`],
  ["int unsigned'(x)", `body[0].value.type: ${CAST}`], ["bit signed'(x)", `body[0].value.type: ${CAST}`],
  ["realtime'(x)", `body[0].value.type: ${CAST}`], ["const'(x)", `body[0].value.type: ${CAST}`],
  ["4'd8'(x)", `body[0].value.type: ${CAST}`],
  ["q.sum(p, i) with (p)", `body[0].value: ${ITERATION}`], ["q.sum with (item)", `body[0].value: ${ITERATION}`],
  ["q.min(p) with (p)", `body[0].value: ${ITERATION}`], ["q.sum(1) with (p)", `body[0].value: ${ITERATION}`],
  ["q.sum(p::x) with (p)", `body[0].value: ${ITERATION}`],
  ["p::f(1)", "body[0].value.callee: a function is called by its name"],
  ["$unit::x", "body[0].value: NameExpression has no counterpart in the SystemVerilog dialect"],
  ["f(.a(1))", "body[0].value.arguments[0]: NamedConnection has no counterpart in the SystemVerilog dialect"],
  ["$bits(int)", "body[0].value.arguments[0]: an argument is an expression"],
  ["f(, 1)", "body[0].value.arguments[0]: an argument is an expression"],
  ["x + (* a *) y", "body[0].value: attributes have no counterpart in the SystemVerilog dialect"],
  ["f (* a *) (1)", "body[0].value: attributes have no counterpart in the SystemVerilog dialect"],
  ["x ? x++ : y", "body[0].value.consequence: IncrementExpression has no counterpart in the SystemVerilog dialect"],
  ["1ns", "body[0].value: TimeLiteral has no counterpart in the SystemVerilog dialect"],
  ["'{1, 2}", "body[0].value: AssignmentPattern has no counterpart in the SystemVerilog dialect"],
  ["tagged a 5", "body[0].value: TaggedExpression has no counterpart in the SystemVerilog dialect"],
  ["super.a", "body[0].value.value: SuperExpression has no counterpart in the SystemVerilog dialect"],
);
const one = new S.IntegerLiteral({ spelling: "1" });
raises(Errors.TranspileError, () => B.term(new S.UnaryExpression({ operator: "++", operand: one })),
  "the operator '++' has no counterpart in the SystemVerilog dialect");
raises(Errors.TranspileError, () => B.term(new S.BinaryExpression({ left: one, operator: "x", right: one })),
  "the operator 'x' has no counterpart in the SystemVerilog dialect");
console.log("ok");

## BRV-06 · The readers take what the writers make: one return, one expression constraint

In [ ]:
const misfits: any = SystemVerilog2023.parse(`function automatic logic f(); endfunction
function automatic void g(); return; endfunction
function automatic int h(); int y = 1; return y; endfunction
function automatic int k(); k = 1; endfunction
class c; constraint a { x > 0; y > 0; } constraint b { soft x > 0; } constraint d { x -> y > 0; } constraint e { } endclass
module m; initial assert (1 == x[0+:2]); endmodule
`);
for (const fn of misfits.items.slice(0, 4)) {
  raises(Errors.TranspileError, () => B.term_of_function(fn), "the function's body must be one return of a value");
}
for (const declaration of misfits.items[4].items) {
  raises(Errors.TranspileError, () => B.term_of_constraint(declaration), "the constraint must hold one expression, not soft");
}
raises(Errors.TranspileError, () => B.term_of_assertion(misfits.items[5].items[0].body),
  "expression.right: a part-select with +: has no counterpart in the dialect");
console.log("ok");

## BRV-07 · Constraints round-trip through the translators, the bridge, the printer and the parser, and agree with Basic where it decides

In [ ]:
const Shipment = new SC.OfObject.Builder().name("Shipment").ref().properties(
  (p: any) => p.name("celsius").of((t: any) => t.as_native(Number)), (p: any) => p.name("hours").of((t: any) => t.as_native(BigInt)),
  (p: any) => p.name("readings").of((t: any) => t.as_indexed((i: any) => i.of((n: any) => n.as_native(Number))))).create();
const store: any = new Proxies.OfStore();
store.register(Shipment);
const shipments = [store.Shipment().celsius(5.5).hours(30n).readings([4.0, 8.6, 9.1]).create(),
  store.Shipment().celsius(9.1).hours(80n).readings([4.0, 12.5]).create(),
  store.Shipment().celsius(3.0).hours(20n).readings([]).create()]; // no readings
const [self, c] = [Basic.variable("this"), Basic.variable("c")];
const CONSTRAINTS = [
  Basic.literal(2.0).le(self.celsius).and_(self.celsius.le(8.0)), self.hours.gt(72n).implies(self.celsius.le(8.0)),
  self.hours.gt(72n).not_(), self.readings.all("c", c.le(12.0)), self.readings.any("c", c.gt(8.0)),
  self.readings.count_where("c", c.gt(8.0)).le(2n), self.readings.count().gt(0n), self.readings.max().le(12.0),
  self.readings.item(0n).ge(2.0), Basic.literal(4.0).in_(self.readings), self.hours.add(10n).mul(2n).sub(1n).gt(100n),
];
const toSv = Translators.between(Basic.DIALECT, E.DIALECT);
const toBasic = Translators.between(E.DIALECT, Basic.DIALECT);
const terms = CONSTRAINTS.map((constraint) => toSv.forward(constraint.data));
const sources = terms.map((term, i) => SystemVerilog2023.print(B.function_(`check${i}`, [], term)));
const checks = SystemVerilog2023.parse(sources.join("\n") + "\n");
CONSTRAINTS.forEach((constraint, i) => {
  const readBack = B.term_of_function(checks.items[i] as S.FunctionDeclaration);
  assert(F.same(readBack, terms[i]) && F.same(toBasic.forward(readBack), constraint.data), sources[i]);
  for (const shipment of shipments) { // where Basic decides, SystemVerilog's rules agree
    const decided = BasicEvaluators.predicate(constraint, shipment);
    if (decided !== null) {
      equal((Evaluators.OfAny(readBack, { this: shipment }) as any).integer(), decided ? 1n : 0n);
    }
  }
});
equal(sources[3], "function automatic logic check3;\n    return this.readings.and(c) with (c <= 12.0);\nendfunction");
console.log(CONSTRAINTS.length, "constraints");